In [2]:
import pandas as pd
import numpy as np
import json
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import shap
import warnings
warnings.filterwarnings('ignore')

In [6]:
DATA_PATH   = 'outputs/features_trend.csv'
OUTPUT_DIR  = 'Rental_Predictor_Data'
TARGET         = 'target_yoy_pct'
CLIP_UPPER     = 25.4
CLIP_LOWER     = -15.0
RANDOM_STATE   = 42

In [16]:
# Load the data and clip
df = pd.read_csv(DATA_PATH, low_memory=False, parse_dates=['date'])
df['zip_code'] = df['zip_code'].astype(str).str.zfill(5)
before_clip = df[TARGET].describe()
df[TARGET] = df[TARGET].clip(lower=CLIP_LOWER, upper=CLIP_UPPER)
print(f"  Rows affected: {((df[TARGET] == CLIP_UPPER) | (df[TARGET] == CLIP_LOWER)).sum():,}")

  Rows affected: 2,141


In [8]:
# Select features

# Categorical features - will be target encoded
CAT_FEATURES = ['State', 'Metro']
 
# Numeric features
NUM_FEATURES = [
    # ZORI lags
    'zori_lag_1m', 'zori_lag_3m', 'zori_lag_6m',
    'zori_lag_12m', 'zori_lag_24m', 'zori_lag_36m',
    # Rolling stats
    'zori_roll_mean_6m', 'zori_roll_mean_12m', 'zori_roll_std_6m',
    # Rate of change
    'zori_mom_pct', 'zori_yoy_pct', 'zori_2y_pct',
    # SAFMR bedroom values
    'fmr_0br', 'fmr_1br', 'fmr_2br', 'fmr_3br', 'fmr_4br',
    # SAFMR lag and YoY
    'fmr_0br_lag1y', 'fmr_1br_lag1y', 'fmr_2br_lag1y', 'fmr_3br_lag1y', 'fmr_4br_lag1y',
    'fmr_0br_yoy_pct', 'fmr_1br_yoy_pct', 'fmr_2br_yoy_pct', 'fmr_3br_yoy_pct', 'fmr_4br_yoy_pct',
    # Bedroom ratios
    'ratio_0br_to_2br', 'ratio_1br_to_2br', 'ratio_3br_to_2br', 'ratio_4br_to_2br',
    # Calendar
    'month', 'quarter', 'year', 'month_sin', 'month_cos',
]
 
# Keep only columns that exist
NUM_FEATURES = [f for f in NUM_FEATURES if f in df.columns]
CAT_FEATURES = [f for f in CAT_FEATURES if f in df.columns]

In [9]:
# Target Encoding for cat features
encoding_maps = {}
for col in CAT_FEATURES:
    means = df.groupby(col)[TARGET].mean()
    encoding_maps[col] = means.to_dict()
    df[f'{col}_encoded'] = df[col].map(means)
    NUM_FEATURES.append(f'{col}_encoded')

In [10]:
# Train / Test Split
X = df[NUM_FEATURES].copy()
y = df[TARGET].copy()
groups = df['zip_code']
 
splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups))
 
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_test = groups.iloc[test_idx]

In [17]:
# Training
print("\nTraining XGBoost...")
model = xgb.XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,   # to prevent overfitting on sparse zip codes
    reg_alpha=0.1,         # L1 regularization
    reg_lambda=1.0,        # L2 regularization
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=50,
    eval_metric='mae',
)
 
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100,
)


Training XGBoost...
[0]	validation_0-mae:4.00989	validation_1-mae:3.87040
[100]	validation_0-mae:2.05908	validation_1-mae:2.09563
[200]	validation_0-mae:1.92364	validation_1-mae:2.02360
[300]	validation_0-mae:1.83243	validation_1-mae:1.99066
[400]	validation_0-mae:1.76330	validation_1-mae:1.97433
[500]	validation_0-mae:1.70090	validation_1-mae:1.95767
[600]	validation_0-mae:1.64696	validation_1-mae:1.94678
[700]	validation_0-mae:1.59733	validation_1-mae:1.93721
[799]	validation_0-mae:1.55236	validation_1-mae:1.93007


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [18]:
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)
 
def eval_metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"  {label}: MAE={mae:.3f}%  RMSE={rmse:.3f}%  R²={r2:.3f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2}
 
train_metrics = eval_metrics(y_train, y_pred_train, "Train")
test_metrics  = eval_metrics(y_test,  y_pred_test,  "Test ")
 
# Per-ZIP test evaluation
df_test = df.iloc[test_idx].copy()
df_test['predicted'] = y_pred_test
df_test['residual']  = df_test[TARGET] - df_test['predicted']
 
zip_eval = df_test.groupby('zip_code').apply(lambda g: pd.Series({
    'mae':   mean_absolute_error(g[TARGET], g['predicted']),
    'rmse':  np.sqrt(mean_squared_error(g[TARGET], g['predicted'])),
    'r2':    r2_score(g[TARGET], g['predicted']) if len(g) > 1 else np.nan,
    'n_obs': len(g),
    'state': g['State'].iloc[0],
    'metro': g['Metro'].iloc[0],
})).reset_index()
 
print(f"\n  Median per-ZIP MAE: {zip_eval['mae'].median():.3f}%")
print(f"  Median per-ZIP R²:  {zip_eval['r2'].median():.3f}")
print(f"\n  Best 5 ZIPs by MAE:")
print(zip_eval.nsmallest(5, 'mae')[['zip_code','state','mae','r2','n_obs']].to_string(index=False))
print(f"\n  Worst 5 ZIPs by MAE:")
print(zip_eval.nlargest(5, 'mae')[['zip_code','state','mae','r2','n_obs']].to_string(index=False))

  Train: MAE=1.552%  RMSE=2.037%  R²=0.872
  Test : MAE=1.930%  RMSE=2.585%  R²=0.778

  Median per-ZIP MAE: 1.806%
  Median per-ZIP R²:  0.641

  Best 5 ZIPs by MAE:
zip_code state      mae        r2  n_obs
   98663    WA 0.653211  0.663143     26
   83402    ID 0.867659  0.138541     26
   55117    MN 0.899002 -0.891809     26
   02169    MA 0.934498  0.902787     91
   28027    NC 0.962567  0.941460     84

  Worst 5 ZIPs by MAE:
zip_code state      mae        r2  n_obs
   46224    IN 6.181881 -9.881134     29
   90265    CA 5.765974 -0.105222     91
   06605    CT 5.074585 -0.744299     34
   30506    GA 4.968008 -0.045497     50
   35205    AL 4.123364 -0.219430     74


In [19]:
# SHAP Analysis
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), min(2000, len(X_test)), replace=False)
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.iloc[sample_idx])
 
shap_importance = pd.DataFrame({
    'feature':    NUM_FEATURES,
    'mean_shap':  np.abs(shap_values).mean(axis=0),
}).sort_values('mean_shap', ascending=False).reset_index(drop=True)
 
print("\n  Top 15 features by SHAP importance:")
print(shap_importance.head(15).to_string(index=False))


  Top 15 features by SHAP importance:
         feature  mean_shap
            year   2.662010
   Metro_encoded   0.798639
     zori_lag_1m   0.394001
   State_encoded   0.350580
           month   0.327316
     zori_2y_pct   0.311946
ratio_4br_to_2br   0.308403
     zori_lag_3m   0.283805
         fmr_0br   0.237802
ratio_3br_to_2br   0.222925
    zori_yoy_pct   0.219933
    zori_mom_pct   0.167997
ratio_0br_to_2br   0.162585
    zori_lag_36m   0.161217
ratio_1br_to_2br   0.156517


In [20]:
# Save Model
model.save_model(f'{OUTPUT_DIR}/model_xgb.json')
 
meta = {
    'features':        NUM_FEATURES,
    'cat_features':    CAT_FEATURES,
    'encoding_maps':   encoding_maps,
    'clip_upper':      CLIP_UPPER,
    'clip_lower':      CLIP_LOWER,
    'train_metrics':   train_metrics,
    'test_metrics':    test_metrics,
    'n_train_rows':    len(X_train),
    'n_test_rows':     len(X_test),
    'n_train_zips':    int(groups.iloc[train_idx].nunique()),
    'n_test_zips':     int(groups_test.nunique()),
}
with open(f'{OUTPUT_DIR}/model_features.json', 'w') as f:
    json.dump(meta, f, indent=2)
 
zip_eval.to_csv(f'{OUTPUT_DIR}/model_eval.csv', index=False)
shap_importance.to_csv(f'{OUTPUT_DIR}/shap_importance.csv', index=False)

In [15]:
# First iteration evaluation
# The good:
# Test MAE and R^2 look good. Train/testing gap is acceptable at 1.55 v 1.93

# The bad:
# Median per zip R^2 is noticably lower than the aggregated number -> Some zips preforming much better than others
# The worst zips have nagative R^2, indicating the model is worse than just predicting means for some markets.
## Possible certain markets behave much differently than the rest of the dataset?
# The best zips by MAE don't always have good R^2
## Model not capturing some patterns well?
# year is the dominant feature by a significant mergin
## The model is learning "What year is it" more than "What are the rent dynamics of this zip code"
### Temporal leakage
# ratio_4br_to_2br and ratio_3br_to_2br ranking higher than other core features is confusing..
## bedroom ratios shouldn't be stronger predictors of aggregated YoY change.
## fmr_0br ranking above fmr_1br is also odd, studio pricing shouldn't be the strongest bedroom signal..


eval_df = pd.read_csv('Rental_Predictor_Data/model_eval.csv')
print(eval_df.nlargest(20, 'mae')[['zip_code','state','metro','mae','r2','n_obs']])
# Below shows a pattern:
## Volatile markets -> Malibu, Naples, Plam Springs, etc
### High end markets where rent swings are potentially driven by additional factors (vacation demand, high net worth migration patterns)
### Can't be learned from ZORI and SAFM alone
## Low observation counts with erratic behavior -> 46224 IN(29 observations), 06605 CT(34 observations), etc
### negative R^2 on these extremely small samples could mean one or two extreme observations are dominating
### Negative R^2 on Indianapolis is a red flag (Data quality issues?)
## Geographic clusters -> Kansas City shows twice, Indianapolis twice, Connecticut twice
### Maybe the model isn't completely capturing the pattern?

     zip_code state                                       metro       mae  \
184     46224    IN            Indianapolis-Carmel-Anderson, IN  6.181881   
337     90265    CA          Los Angeles-Long Beach-Anaheim, CA  5.765974   
11       6605    CT             Bridgeport-Stamford-Norwalk, CT  5.074585   
112     30506    GA                             Gainesville, GA  4.968008   
159     35205    AL                       Birmingham-Hoover, AL  4.123364   
193     52807    IA         Davenport-Moline-Rock Island, IA-IL  4.121351   
182     46214    IN            Indianapolis-Carmel-Anderson, IN  3.960966   
284     79606    TX                                 Abilene, TX  3.931391   
402     95814    CA             Sacramento-Roseville-Folsom, CA  3.844998   
152     34109    FL                     Naples-Marco Island, FL  3.796326   
362     92262    CA        Riverside-San Bernardino-Ontario, CA  3.746824   
252     75226    TX             Dallas-Fort Worth-Arlington, TX  3.739282   

In [ ]:
# Thoughts for next iteration:
# Remove year as a feature, temporal signal should come entirely from the lage and rate of change features
# Drop zip codes with < 36 observations
# Add a new is_luxury flag. Logic -> if median ZORI > 90th percentile nationally.
# Consider removal of bedroom ratio columns as they might be acting as noisy proxies for market type.